# Part 1 Data Cleaning and Validation

After analyzing all requirements, here is how I will structure the work:

Principles

- The raw table is never modified. Everything is read from ohlc_hw and written to new tables.
- Every raw row ends up in exactly one place: retained, removed (exact duplicate or all prices missing), or quarantined with a reason. 
- Repairs are flagged, never silent. Each repair gets a boolean flag column (e.g. flag_volume_filled, flag_ohlc_repaired, flag_adj_reconstructed, flag_extreme_return_kept).
- Original labels are preserved (raw_ticker), along with a ticker-normalization audit map.

Pipeline order (the order matters, since several steps depend on earlier ones)

- Load and normalize keys. Strip markup and whitespace and uppercase tickers, then parse ts to ISO YYYY-MM-DD. This happens before any grouping or deduplication.
- Profile (diagnostics only).Sort by (ticker, ts), and then
    - (low severity) exact dupli cate  
    - (high severity) conflicting key
    - (low severity) ticker format, ticker label that needed normalizing
    - ('medium' severity) Check missing cells per column, 
    - ('low' severity) missing trading dates versus the ohlc calendar (with a summary: "Ticker X is missing Y trading days between [start] and [end] dates"), 
    - ('medium' severity) impossible OHLC relationships, 
    - ('high' severity) non-positive prices, 
    - ('high' severity) flag extreme returns for manual checks, 
    - ('high' severity) flag zero volume days for manual checks.
- Generate a pandas dataframe 'Data Quality Report' based on the Profile (diagnostics only) step. The summary counts number of data issues per category. Column names are ticker | issue_type | issue_date | description | severity (high/medium/low).
- Deduplicate. Drop exact duplicates. For conflicting (ticker, ts) keys, resolve using evidence (the ohlc reference and OHLC validity), and quarantine the key if the evidence is inconclusive.
- Drop rows with all four raw prices missing, before any filling.
- Sort by (ticker, ts), then repair raw prices. Invalid relationships become NaN, then are Interpolate using other prices (open and close, for high and low). The OHLC bounds are rechecked afterwards, since a stale fill can violate today’s range. Leading gaps with nothing to carry forward are quarantined.
- Volume. Missing volume is filled with 0 and flagged.
- Extreme returns. Compute per-ticker z-scores as you specified based on repaired prices, then classify each hit:
    - Split-like: the volume jumps by a near-rational ratio
    - Data error: the move disagrees with the ohlc reference.
    - Inconclusive: quarantine and flag with a stated uncertainty.
- Only confirmed errors are interpolated, and only between valid neighbors.
- Missing Adjusted close. Take a date-specific factor adj_close / close from the matched ohlc row, then rebuild adj_close = clean_close × factor. There is no forward-filling of factors, and unresolved rows are quarantined and counted.
- Final validation. Check unique keys, finite positive prices, finite nonnegative volume, OHLC bounds, and adjusted-price consistency.
- Show before/after stats. 
- Write outputs in data.db: ohlc_hw_cleaned, also ohlc_hw_quarantine (with reason), ohlc_hw_removed (with reason) and ohlc_hw_ticker_audit

## 1.1 Load and Inspect the Data

In [32]:
# Load the `ohlc_hw` table from `data/data.db`

import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("data/data.db")

df = pd.read_sql("SELECT * from ohlc_hw", conn)
df_calendar = pd.read_sql("SELECT * from ohlc", conn)
conn.close()

display(df.head())
display(df_calendar.head())

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-10-14,161.40,162.14,160.90,161.60,161.60,5964000,JNJ
1,2024-10-11,162.18,162.66,161.20,161.46,161.46,5253500,JNJ
2,2024-10-10,161.36,161.39,160.00,160.51,160.51,4793900,JNJ
3,2024-09-30,161.57,162.18,160.68,162.06,162.06,6010407,JNJ
4,2024-09-27,162.00,163.43,161.34,161.40,161.40,7553905,JNJ


In [33]:
# Display basic statistics (`.info()`, `.describe()`)

df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


**<u>My Observation</u>**

*Hypothesis: only adj_close has missing values*

**Identify the columns and their meanings**

This table contains daily OHLC (Open, High, Low, Close) data for multiple stocks.
The columns are ts (observation date), ticker, raw open, high, low, close, vendor adj_close, and volume


In [34]:
# Show the date range and list of tickers
ts_col = pd.to_datetime(df['ts'])

#date range
print("date range in ohlc_hw")
ts_col.min(), ts_col.max()

date range in ohlc_hw


(Timestamp('1962-01-02 00:00:00'), Timestamp('2024-12-06 00:00:00'))

In [35]:
#compare with reference table ohlc
ts_col2 = pd.to_datetime(df_calendar['ts'])

#date range
print("date range in ohlc")
ts_col2.min(), ts_col2.max()

date range in ohlc


(Timestamp('1962-01-02 00:00:00'), Timestamp('2024-12-06 00:00:00'))

In [36]:
#list of tickers (sorted)
ticker_col = df['ticker']
sorted(ticker_col.unique())

['<a target="_blank"><span style="color: blue">AAPL</span></a>',
 '<a target="_blank"><span style="color: blue">AMGN</span></a>',
 '<a target="_blank"><span style="color: blue">AMZN</span></a>',
 '<a target="_blank"><span style="color: blue">AXP</span></a>',
 '<a target="_blank"><span style="color: blue">BA</span></a>',
 '<a target="_blank"><span style="color: blue">CAT</span></a>',
 '<a target="_blank"><span style="color: blue">CRM</span></a>',
 '<a target="_blank"><span style="color: blue">CSCO</span></a>',
 '<a target="_blank"><span style="color: blue">CVX</span></a>',
 '<a target="_blank"><span style="color: blue">DIS</span></a>',
 '<a target="_blank"><span style="color: blue">GS</span></a>',
 '<a target="_blank"><span style="color: blue">HD</span></a>',
 '<a target="_blank"><span style="color: blue">HON</span></a>',
 '<a target="_blank"><span style="color: blue">IBM</span></a>',
 '<a target="_blank"><span style="color: blue">JNJ</span></a>',
 '<a target="_blank"><span style="color

In [37]:
# Load and normalize keys. Strip markup and whitespace and uppercase tickers, then parse ts to ISO YYYY-MM-DD.

def normalize_keys(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Normalize the (ticker, ts) key columns before any grouping or dedup.

    * ticker: remove HTML tags (e.g. '<a ...><span ...>SHW</span></a>' -> 'SHW'),
      strip whitespace, upper-case.
    * ts: parse to a date and re-emit as ISO 'YYYY-MM-DD' strings so SQL date
      filters and key comparisons behave consistently.
    """
    missing = {"ticker", "ts"} - set(raw.columns)
    if missing:
        raise KeyError(f"raw data is missing required columns: {sorted(missing)}")

    df = raw.copy()
    df["raw_ticker"], df["raw_ts"] = df["ticker"], df["ts"]

    df["ticker"] = (
        df["raw_ticker"].astype("string")
        .str.replace(r"<[^>]+>", "", regex=True)  # drop every HTML tag, keep inner text
        .str.strip()
        .str.upper()
        .mask(lambda s: s.eq(""))                 # label that was only markup -> missing
    )

    # errors="coerce" turns unparseable dates into NaT instead of aborting the load.
    df["ts"] = pd.to_datetime(df["raw_ts"], errors="coerce", format="mixed").dt.strftime("%Y-%m-%d")

    # The quarantine step reads this column to explain why a key is unusable.
    problems = pd.DataFrame({
        "unusable ticker": df["ticker"].isna(),
        "unparseable ts": df["ts"].isna(),
    })
    df["key_issue"] = problems.apply(lambda r: "; ".join(r.index[r]) or None, axis=1)

    ticker_audit = (
        df.groupby(["raw_ticker", "ticker"], dropna=False).size()
        .rename("n_rows").reset_index()
        .sort_values(["ticker", "raw_ticker"], na_position="last", ignore_index=True)
    )
    # A missing label counts as changed so it appears in the audit review.
    ticker_audit["was_changed"] = (
        ticker_audit["raw_ticker"].astype("string")
        .ne(ticker_audit["ticker"].astype("string"))
        .fillna(True)
        .astype(bool)
    )
    return df, ticker_audit

In [38]:
# before and after ticker normalization
print("Unique tickers before normailization:", df["ticker"].nunique())
df, ticker_audit = normalize_keys(df)
print("Unique tickers after normalization: ", df["ticker"].nunique())

Unique tickers before normailization: 150
Unique tickers after normalization:  30


In [54]:
# Missing in the raw columns vs. unusable after normalization
print("raw_ticker missing:", df["raw_ticker"].isna().sum())
print("raw_ts missing:    ", df["raw_ts"].isna().sum())
print("ticker missing after normalization:", df["ticker"].isna().sum())
print("ts missing after normalization:    ", df["ts"].isna().sum())

# Breakdown of rows with an unusable key, and the rows themselves
print(df["key_issue"].value_counts(dropna=False))
display(df[df["key_issue"].notna()].head())

raw_ticker missing: 0
raw_ts missing:     0
ticker missing after normalization: 0
ts missing after normalization:     0
key_issue
None    397711
Name: count, dtype: int64


,ts,open,high,low,close,adj_close,volume,ticker,raw_ticker,raw_ts,key_issue


## 1.2 Detect Data Quality Issues

In [39]:
KEY = ["ticker", "ts"]
PRICE_COLS = ["open", "high", "low", "close"]
VALUE_COLS = PRICE_COLS + ["adj_close", "volume"]
REPORT_COLS = ["ticker", "issue_type", "issue_date", "description", "severity"]

# Single source of truth for severities, so every check and the summary agree.
SEVERITY = {
    "exact_duplicate": "low",
    "conflicting_key": "high",
    "ticker_format": "low",
    "missing_value": "medium",
    "missing_date": "low",
    "impossible_ohlc": "medium",
    "non_positive_price": "high",
    "extreme_return": "high",
    "zero_volume": "high",
}

df = df.sort_values(KEY, kind="stable", ignore_index=True)

# Rows with an unusable key cannot be located in the report, so checks skip them;
# the quarantine step deals with them later.
usable = df[df["key_issue"].isna()]
# Exact duplicates are dropped for diagnostics only (df is untouched), so one
# repeated record is not counted once per copy by the value checks below.
diag = usable.drop_duplicates(subset=KEY + VALUE_COLS)

dq_issues = {}  # issue_type -> report rows, assembled into the report in the last cell


def empty_report() -> pd.DataFrame:
    return pd.DataFrame(columns=REPORT_COLS)


def make_issues(ticker, issue_date, issue_type, description) -> pd.DataFrame:
    """Build Data Quality Report rows; scalar arguments broadcast over the Series ones."""
    out = pd.DataFrame({"ticker": ticker})
    out["issue_type"] = issue_type
    out["issue_date"] = issue_date
    out["description"] = description
    out["severity"] = SEVERITY[issue_type]
    return out[REPORT_COLS].reset_index(drop=True)


print(f"{len(df)} rows | {len(df) - len(usable)} with unusable key | "
      f"{len(usable) - len(diag)} exact duplicates | {len(diag)} rows used for value checks")

397711 rows | 0 with unusable key | 29460 exact duplicates | 368251 rows used for value checks


In [40]:
def check_exact_duplicates(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag every extra copy of a row identical in key and all value columns."""
    # keep="first": the first occurrence is the record, later copies are the issue.
    dups = rows[rows.duplicated(KEY + VALUE_COLS, keep="first")]
    return make_issues(dups["ticker"], dups["ts"], "exact_duplicate",
                       "Identical to an earlier row (after ticker normalization)")


dq_issues["exact_duplicate"] = check_exact_duplicates(usable)
print(len(dq_issues["exact_duplicate"]), "exact duplicate rows")
display(dq_issues["exact_duplicate"].head())

29460 exact duplicate rows


,ticker,issue_type,issue_date,description,severity
0,AAPL,exact_duplicate,1981-01-02,Identical to an earlier row (after ticker norm...,low
1,AAPL,exact_duplicate,1981-01-05,Identical to an earlier row (after ticker norm...,low
2,AAPL,exact_duplicate,1981-02-10,Identical to an earlier row (after ticker norm...,low
3,AAPL,exact_duplicate,1981-02-27,Identical to an earlier row (after ticker norm...,low
4,AAPL,exact_duplicate,1981-03-05,Identical to an earlier row (after ticker norm...,low


In [41]:
def check_conflicting_keys(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag (ticker, ts) keys that still hold different values once exact duplicates are gone.

    * One issue per key, naming the columns that disagree, so manual review knows where to look.
    * `rows` must already be free of exact duplicates.
    """
    conflicts = rows[rows.duplicated(KEY, keep=False)]
    if conflicts.empty:
        return empty_report()
    by_key = conflicts.groupby(KEY)
    n_rows = by_key.size().rename("n_rows")
    disagree = by_key[VALUE_COLS].nunique(dropna=False).gt(1)
    cols = disagree.apply(lambda r: ", ".join(r.index[r]), axis=1).rename("cols")
    out = pd.concat([n_rows, cols], axis=1).reset_index()
    desc = out["n_rows"].astype(str) + " rows share this key but differ in: " + out["cols"]
    return make_issues(out["ticker"], out["ts"], "conflicting_key", desc)


dq_issues["conflicting_key"] = check_conflicting_keys(diag)
print(len(dq_issues["conflicting_key"]), "conflicting keys")
display(dq_issues["conflicting_key"].head())

0 conflicting keys


,ticker,issue_type,issue_date,description,severity


In [42]:
def check_ticker_format(audit: pd.DataFrame) -> pd.DataFrame:
    """One issue per raw ticker label that needed normalizing (from the audit map)."""
    changed = audit[audit["was_changed"]]
    desc = pd.Series(
        [f"Label {raw!r} ({n} rows) " + (f"normalized to {tkr!r}" if pd.notna(tkr) else "left no usable ticker")
         for raw, tkr, n in zip(changed["raw_ticker"], changed["ticker"], changed["n_rows"])],
        index=changed.index, dtype=object,
    )
    # A label-level issue has no single date, so issue_date stays empty.
    return make_issues(changed["ticker"], None, "ticker_format", desc)


dq_issues["ticker_format"] = check_ticker_format(ticker_audit)
print(len(dq_issues["ticker_format"]), "labels needed normalizing")
display(dq_issues["ticker_format"].head())

120 labels needed normalizing


,ticker,issue_type,issue_date,description,severity
0,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""color: ...",low
1,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""color: ...",low
2,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""font-we...",low
3,AAPL,ticker_format,None,"Label '<a target=""_blank""><span style=""font-we...",low
4,AMGN,ticker_format,None,"Label '<a target=""_blank""><span style=""color: ...",low


In [43]:
def check_missing_cells(rows: pd.DataFrame) -> pd.DataFrame:
    """One issue per missing cell in the price, adj_close and volume columns."""
    long = rows.melt(id_vars=KEY, value_vars=VALUE_COLS, var_name="column")
    missing = long[long["value"].isna()]
    return make_issues(missing["ticker"], missing["ts"], "missing_value", "Missing " + missing["column"])


print("Missing cells per column:")
display(diag[VALUE_COLS].isna().sum().to_frame("n_missing"))  # tests the "only adj_close is missing" hypothesis

dq_issues["missing_value"] = check_missing_cells(diag)
print(len(dq_issues["missing_value"]), "missing cells")

Missing cells per column:


,n_missing
open,0
high,0
low,0
close,0
adj_close,18374
volume,0


18374 missing cells


In [44]:
def check_missing_dates(rows: pd.DataFrame, calendar_ts: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare each ticker's dates with the trading days in the ohlc reference table.

    * The window is the ticker's own first-to-last date in ohlc_hw, so a ticker that
      starts late or ends early is not charged for dates outside its history.
    * Returns (report rows, one summary row per ticker).
    """
    calendar = pd.Index(sorted(pd.to_datetime(calendar_ts).dt.strftime("%Y-%m-%d").dropna().unique()))
    frames = []
    for ticker, g in rows.groupby("ticker"):
        seen = pd.Index(g["ts"].unique())
        start, end = seen.min(), seen.max()
        in_window = calendar[(calendar >= start) & (calendar <= end)]
        missing = in_window.difference(seen)
        if len(missing):
            frames.append(pd.DataFrame({"ticker": ticker, "ts": missing, "start": start, "end": end}))
    if not frames:
        return empty_report(), pd.DataFrame(columns=["ticker", "n_missing", "start", "end"])

    gaps = pd.concat(frames, ignore_index=True)
    summary = (gaps.groupby("ticker")
               .agg(n_missing=("ts", "size"), start=("start", "first"), end=("end", "first"))
               .reset_index())
    report = make_issues(gaps["ticker"], gaps["ts"], "missing_date",
                         "Trading day in the ohlc calendar has no row in ohlc_hw")
    return report, summary


dq_issues["missing_date"], missing_date_summary = check_missing_dates(diag, df_calendar["ts"])

for r in missing_date_summary.itertuples():
    print(f"Ticker {r.ticker} is missing {r.n_missing} trading days between {r.start} and {r.end}")
print(len(dq_issues["missing_date"]), "missing ticker-dates in total")

Ticker AAPL is missing 6 trading days between 1980-12-12 and 2024-10-18
Ticker AXP is missing 25 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 50 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker CVX is missing 25 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 1 trading days between 1962-01-02 and 2024-12-06
Ticker HON is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 25 trading days between 1962-01-02 and 2024-12-06
Ticker JNJ is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker JPM is missing 26 trading days between 1969-03-05 and 2024-10-18
Ticker KO is missing 26 trading days between 1962-01-02 and 2024-12-06
Ticker MCD is missing 26 trading days between 1966-07-05 and 2024-10-18
Ticker MMM is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker MRK is missing 26 trading days between 1962-01-02 and 2024-1

In [45]:
def check_impossible_ohlc(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag rows that break the OHLC relationships.

    * high must be >= max(open, close) and >= low; low must be <= min(open, close).
    * A NaN cell never violates a rule (it is reported as missing_value instead).
    """
    # skipna: if open is missing, close alone sets the bound.
    body_top = rows[["open", "close"]].max(axis=1)
    body_bottom = rows[["open", "close"]].min(axis=1)
    rules = pd.DataFrame({
        "high < max(open, close)": rows["high"] < body_top,
        "low > min(open, close)": rows["low"] > body_bottom,
        "high < low": rows["high"] < rows["low"],
    })
    bad = rules.any(axis=1)
    if not bad.any():
        return empty_report()
    desc = rules[bad].apply(lambda r: "; ".join(r.index[r]), axis=1)  # which rules failed
    return make_issues(rows.loc[bad, "ticker"], rows.loc[bad, "ts"], "impossible_ohlc", desc)


dq_issues["impossible_ohlc"] = check_impossible_ohlc(diag)
print(len(dq_issues["impossible_ohlc"]), "rows with impossible OHLC relationships")
display(dq_issues["impossible_ohlc"].head())

5 rows with impossible OHLC relationships


,ticker,issue_type,issue_date,description,severity
0,CVX,impossible_ohlc,1962-01-02,"low > min(open, close)",medium
1,JNJ,impossible_ohlc,1962-01-02,"low > min(open, close)",medium
2,MCD,impossible_ohlc,1966-07-05,"low > min(open, close)",medium
3,MRK,impossible_ohlc,1962-01-02,"low > min(open, close)",medium
4,PG,impossible_ohlc,1962-01-02,"low > min(open, close)",medium


In [46]:
def check_non_positive_prices(rows: pd.DataFrame) -> pd.DataFrame:
    """One issue per open/high/low/close/adj_close cell that is <= 0 or infinite."""
    long = rows.melt(id_vars=KEY, value_vars=PRICE_COLS + ["adj_close"], var_name="column")
    bad = long[long["value"].le(0) | np.isinf(long["value"])]  # NaN is not flagged here
    desc = "Non-positive/non-finite " + bad["column"] + " = " + bad["value"].astype(str)
    return make_issues(bad["ticker"], bad["ts"], "non_positive_price", desc)


dq_issues["non_positive_price"] = check_non_positive_prices(diag)
print(len(dq_issues["non_positive_price"]), "non-positive price cells")
display(dq_issues["non_positive_price"].head())

5 non-positive price cells


,ticker,issue_type,issue_date,description,severity
0,CVX,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high
1,JNJ,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high
2,MCD,non_positive_price,1966-07-05,Non-positive/non-finite open = 0.0,high
3,MRK,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high
4,PG,non_positive_price,1962-01-02,Non-positive/non-finite open = 0.0,high


In [47]:
def check_extreme_returns(rows: pd.DataFrame, n_std: float = 3) -> pd.DataFrame:
    """Flag closes whose daily return is more than n_std std away from the ticker's mean return.

    * Rule: abs(ret - mean_ret) > n_std * std_ret per ticker (sample std, ddof=1).
    * Returns skip missing and non-positive closes (a return spans the gap) and
      unresolved conflicting keys, so one bad row cannot poison the chain.
    * Diagnostic only: the cleaning step recomputes this on repaired prices.
    """
    clean = ~rows.duplicated(KEY, keep=False) & rows["close"].gt(0)
    s = rows.loc[clean, KEY + ["close"]].sort_values(KEY)

    s["prev_ts"] = s.groupby("ticker")["ts"].shift()
    s["ret"] = s["close"] / s.groupby("ticker")["close"].shift() - 1
    s["dev"] = s["ret"] - s.groupby("ticker")["ret"].transform("mean")
    s["std"] = s.groupby("ticker")["ret"].transform("std")

    hits = s[s["dev"].abs() > n_std * s["std"]]  # NaN std (too few returns) never flags
    if hits.empty:
        return empty_report()
    desc = hits.apply(
        lambda r: f"close {r['prev_ts']} -> {r['ts']}: return {r['ret']:+.1%} "
                  f"({r['dev'] / r['std']:+.1f} std from ticker mean)", axis=1)
    return make_issues(hits["ticker"], hits["ts"], "extreme_return", desc)


dq_issues["extreme_return"] = check_extreme_returns(diag)
print(len(dq_issues["extreme_return"]), "extreme returns to review")
display(dq_issues["extreme_return"].head())

5056 extreme returns to review


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,close 1980-12-24 -> 1980-12-26: return +9.2% (...,high
1,AAPL,extreme_return,1981-04-02,close 1981-04-01 -> 1981-04-02: return +8.8% (...,high
2,AAPL,extreme_return,1981-09-25,close 1981-09-24 -> 1981-09-25: return -13.0% ...,high
3,AAPL,extreme_return,1982-03-19,close 1982-03-18 -> 1982-03-19: return +9.0% (...,high
4,AAPL,extreme_return,1982-11-22,close 1982-11-19 -> 1982-11-22: return -8.9% (...,high


In [49]:
def check_zero_volume(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag days with volume == 0. Missing volume is a separate issue (missing_value)."""
    zero = rows[rows["volume"].eq(0)]
    return make_issues(zero["ticker"], zero["ts"], "zero_volume", "Volume is zero: review manually")


dq_issues["zero_volume"] = check_zero_volume(diag)
print(len(dq_issues["zero_volume"]), "zero-volume days")
display(dq_issues["zero_volume"].head())

75 zero-volume days


,ticker,issue_type,issue_date,description,severity
0,AXP,zero_volume,1972-12-14,Volume is zero: review manually,high
1,HON,zero_volume,1982-09-23,Volume is zero: review manually,high
2,HON,zero_volume,1982-09-24,Volume is zero: review manually,high
3,IBM,zero_volume,1973-10-09,Volume is zero: review manually,high
4,IBM,zero_volume,1975-01-27,Volume is zero: review manually,high


In [52]:
# Data Quality Report and summary

SEVERITY_ORDER = ["high", "medium", "low"]

frames = [f for f in dq_issues.values() if not f.empty]
dq_report = pd.concat(frames, ignore_index=True)
dq_report["severity"] = pd.Categorical(dq_report["severity"], categories=SEVERITY_ORDER, ordered=True)
dq_report = dq_report.sort_values(["severity", "issue_type", "ticker", "issue_date"], ignore_index=True)

# Summary lists every check, including those with zero hits, so it is clear each one ran.
dq_summary = (
    pd.DataFrame({
        "issue_type": list(dq_issues),
        "severity": [SEVERITY[k] for k in dq_issues],
        "n_issues": [len(v) for v in dq_issues.values()],
    })
    .assign(severity=lambda d: pd.Categorical(d["severity"], categories=SEVERITY_ORDER, ordered=True))
    .sort_values(["severity", "n_issues"], ascending=[True, False], ignore_index=True)
)

display(dq_summary)
print("Total issues:", len(dq_report))
display(dq_report.head(20))

,issue_type,severity,n_issues
0,extreme_return,high,5056
1,zero_volume,high,75
2,non_positive_price,high,5
3,conflicting_key,high,0
4,missing_value,medium,18374
5,impossible_ohlc,medium,5
6,exact_duplicate,low,29460
7,missing_date,low,445
8,ticker_format,low,120


Total issues: 53540


,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26,close 1980-12-24 -> 1980-12-26: return +9.2% (...,high
1,AAPL,extreme_return,1981-04-02,close 1981-04-01 -> 1981-04-02: return +8.8% (...,high
2,AAPL,extreme_return,1981-09-25,close 1981-09-24 -> 1981-09-25: return -13.0% ...,high
3,AAPL,extreme_return,1982-03-19,close 1982-03-18 -> 1982-03-19: return +9.0% (...,high
4,AAPL,extreme_return,1982-11-22,close 1982-11-19 -> 1982-11-22: return -8.9% (...,high
5,AAPL,extreme_return,1982-11-30,close 1982-11-29 -> 1982-11-30: return +10.4% ...,high
6,AAPL,extreme_return,1983-01-20,close 1983-01-19 -> 1983-01-20: return +11.2% ...,high
7,AAPL,extreme_return,1983-04-20,close 1983-04-19 -> 1983-04-20: return +8.9% (...,high
8,AAPL,extreme_return,1983-07-26,close 1983-07-25 -> 1983-07-26: return -9.3% (...,high
9,AAPL,extreme_return,1983-08-31,close 1983-08-30 -> 1983-08-31: return +13.3% ...,high


In [55]:
print(list(df.columns))

['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker', 'raw_ticker', 'raw_ts', 'key_issue']
